In [1]:
import duckdb
from pathlib import Path
conn = duckdb.connect(Path(r"C:\Users\eddiec11us\dev_apps\customer-matching-app\src\data\db.duckdb"))

In [ ]:
query = """
WITH base AS (
SELECT
  vc.vendor_customer_id,
  vc.vendor_name,
  vc.normalized_vendor_customer_name,
  vc.first3_token,
  vc.billing_state,
  vc.normalized_billing_zip,
  p.parent_account_name
FROM vendor_customers vc

LEFT JOIN vendor_customer_to_parent_account_map pid ON 
  vc.vendor_customer_id = pid.vendor_customer_id
LEFT JOIN parent_accounts p ON
  pid.parent_account_id = p.parent_account_id

WHERE 
  vc.first3_token IS NOT NULL
  AND vc.billing_state IS NOT NULL
  AND vc.normalized_billing_zip IS NOT NULL
), counted AS (
SELECT 
  COUNT(*) OVER (
  PARTITION BY base.first3_token, base.billing_state
  ) AS sibling_count
FROM base
), ranked AS (
SELECT 
  *,
  DENSE_RANK() OVER (
  ORDER BY base.first3_token, base.billing_state
  ) AS group_index,
FROM counted
)
SELECT * FROM ranked
WHERE sibling_count > 1
ORDER BY group_index ASC
"""

In [40]:
df = conn.sql(query=query).df()
df

BinderException: Binder Error: Referenced table "base" not found!
Candidate tables: "sibling_count"